# Crawl Dữ Liệu Việc Làm CNTT từ TopDev (Mẫu 50 Jobs)

> **URL tìm kiếm**: `https://topdev.vn/viec-lam/tim-kiem?job_categories_ids=2%2C3%2C4%2C5%2C6%2C7%2C8%2C9%2C10%2C11%2C12%2C13%2C67`
>
> **Kỹ thuật trích xuất**:
> 1. **Vượt cơ chế bảo vệ**: Dùng `curl_cffi` giả lập Chrome (`impersonate="chrome124"`).
> 2. **Danh sách việc làm**: Trang tìm kiếm TopDev xây dựng trên Next.js App Router (React Server Component streaming). Code bóc tách các job slug từ payload RSC stream (`self.__next_f.push`).
> 3. **Chi tiết việc làm**: TopDev nhúng dữ liệu chuẩn cấu trúc `application/ld+json` (chuẩn `@type: JobPosting` của Schema.org), cung cấp đầy đủ thông tin chuẩn xác về tiêu đề, công ty, mức lương, kỹ năng, địa điểm và mô tả công việc.

In [ ]:
import os
import re
import json
import time
import random
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests
from tqdm import tqdm

## Phần 1: Crawl Danh Sách Việc Làm (TopDev Job Listings)

In [ ]:
def crawl_topdev_jobs(target_count: int = 50, delay_range: tuple = (1.2, 2.0)) -> pd.DataFrame:
    """
    Crawl danh sách công việc từ trang tìm kiếm TopDev theo các category IT.
    Bóc tách các job slug từ React Server Component (RSC) streaming của Next.js.
    """
    session = requests.Session(impersonate="chrome124")
    base_url = "https://topdev.vn/viec-lam/tim-kiem?job_categories_ids=2%2C3%2C4%2C5%2C6%2C7%2C8%2C9%2C10%2C11%2C12%2C13%2C67&page={page}"
    
    all_jobs = []
    seen_ids = set()
    page = 1
    
    print(f"[*] Bắt đầu lấy danh sách URL từ TopDev (Mục tiêu: {target_count} jobs)...")
    
    with tqdm(total=target_count, desc="Thu thập link TopDev") as pbar:
        while len(all_jobs) < target_count and page <= 10:
            url = base_url.format(page=page)
            try:
                resp = session.get(url, timeout=20)
                if resp.status_code == 200:
                    pushes = re.findall(r'self\.__next_f\.push\(\[1,"(.*?)"\]\)', resp.text, re.DOTALL)
                    decoded = "".join(pushes).encode('utf-8').decode('unicode_escape', errors='ignore')
                    matches = re.findall(r'/viec-lam/([a-zA-Z0-9\-_]+-(\d{5,8}))', decoded)
                    
                    for slug, jid in matches:
                        if jid not in seen_ids:
                            seen_ids.add(jid)
                            all_jobs.append({
                                "source": "topdev",
                                "source_job_id": jid,
                                "job_url": f"https://topdev.vn/viec-lam/{slug}",
                                "company_logo_url": "",
                                "tile_raw": "",
                                "company_name_raw": "",
                                "location_raw": "",
                                "salary_raw": "",
                                "experience_raw": "",
                                "posted_at_raw": "",
                                "crawled_at": datetime.now().isoformat()
                            })
                            pbar.update(1)
                            if len(all_jobs) >= target_count:
                                break
                else:
                    print(f"[!] Page {page} lỗi HTTP {resp.status_code}")
                    break
            except Exception as e:
                print(f"[!] Lỗi khi request page {page}: {e}")
                break
                
            page += 1
            time.sleep(random.uniform(*delay_range))
            
    df = pd.DataFrame(all_jobs)
    print(f"[+] Đã thu thập được {len(df)} link việc làm từ TopDev.")
    return df

In [ ]:
df_topdev_jobs = crawl_topdev_jobs(target_count=50)

# Lưu danh sách ra thư mục data/raw
output_dir = os.path.join("..", "data", "raw")
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "topdev_jobs_raw.csv")
json_path = os.path.join(output_dir, "topdev_jobs_raw.json")

df_topdev_jobs.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_topdev_jobs.to_json(json_path, orient="records", force_ascii=False, indent=2)

print(f"[+] Đã lưu {len(df_topdev_jobs)} jobs vào {csv_path} và {json_path}")
df_topdev_jobs.head(3)

## Phần 2: Crawl Chi Tiết Từng Job (TopDev Job Details)

Bóc tách toàn bộ thông tin chi tiết qua `JobPosting` LD+JSON và HTML của trang chi tiết.

In [ ]:
def clean_html(raw_html: str) -> str:
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text(separator="\n", strip=True)

def parse_topdev_detail(session, job_item: dict) -> dict:
    url = job_item["job_url"]
    job_id = str(job_item["source_job_id"])
    
    resp = session.get(url, timeout=20)
    if resp.status_code != 200:
        return None
        
    soup = BeautifulSoup(resp.text, "html.parser")
    ld_data = {}
    for s in soup.find_all("script", type="application/ld+json"):
        try:
            d = json.loads(s.string)
            if d.get("@type") == "JobPosting":
                ld_data = d
                break
        except Exception:
            pass
            
    title = ld_data.get("title") or ""
    if not title and soup.title:
        title = soup.title.string.split("|")[0].replace("Tuyển dụng", "").strip()
        
    hiring_org = ld_data.get("hiringOrganization") or {}
    comp_name = hiring_org.get("name", "")
    comp_url = hiring_org.get("sameAs", "")
    
    # Salary
    salary_info = ld_data.get("baseSalary") or {}
    val_info = salary_info.get("value") or {}
    if isinstance(val_info, dict):
        sal_val = val_info.get("value")
        if sal_val == "Negotiable":
            salary_raw = "Thương lượng"
        elif "minValue" in val_info or "maxValue" in val_info:
            min_v = val_info.get("minValue", "")
            max_v = val_info.get("maxValue", "")
            salary_raw = f"{min_v} - {max_v} {salary_info.get('currency', 'VND')}"
        else:
            salary_raw = str(sal_val or "Thương lượng")
    else:
        salary_raw = str(val_info or "Thương lượng")
        
    # Location
    job_loc = ld_data.get("jobLocation") or {}
    addr = job_loc.get("address") or {}
    loc_parts = []
    if addr.get("streetAddress"): loc_parts.append(addr.get("streetAddress"))
    if addr.get("addressLocality"): loc_parts.append(addr.get("addressLocality"))
    if addr.get("addressRegion"): loc_parts.append(addr.get("addressRegion"))
    location_raw = ", ".join(loc_parts) if loc_parts else (addr.get("addressRegion") or "")
    
    # Skills & dates
    skills_raw = str(ld_data.get("skills") or "")
    posted_at = str(ld_data.get("datePosted") or "")
    deadline = str(ld_data.get("validThrough") or "")
    
    # Description & requirements
    desc_html = ld_data.get("description", "")
    desc_text = clean_html(desc_html)
    
    b = ld_data.get("jobBenefits")
    benefits_raw = clean_html(b) if isinstance(b, str) else ("\n".join(b) if isinstance(b, list) else "")
    exp_req = ld_data.get("experienceRequirements", "")
    exp_raw = clean_html(exp_req) if isinstance(exp_req, str) else ""
    emp_type = str(ld_data.get("employmentType") or "")
    industry = str(ld_data.get("industry") or "Information Technology")
    
    return {
        "detail_id": f"topdev_{job_id}",
        "source": "topdev",
        "source_job_id": job_id,
        "job_url": url,
        "title_raw": title,
        "company_name_raw": comp_name,
        "company_url": comp_url,
        "company_size_raw": "",
        "company_location_raw": location_raw,
        "industry_raw": industry,
        "salary_raw": salary_raw,
        "experience_raw": exp_raw,
        "education_raw": "",
        "location_raw": location_raw,
        "work_mode_raw": "",
        "employment_type_raw": emp_type,
        "working_time_raw": "",
        "overview_raw": "",
        "responsibilities_raw": desc_text,
        "requirements_raw": exp_raw,
        "benefits_raw": benefits_raw,
        "required_skills_raw": skills_raw,
        "preferred_skills_raw": "",
        "published_at_raw": posted_at,
        "updated_at_raw": "",
        "deadline_raw": deadline,
        "crawled_at": datetime.now().isoformat()
    }

def crawl_all_topdev_details(input_json_path: str, output_json_path: str, output_csv_path: str, delay_range=(1.2, 2.0)) -> pd.DataFrame:
    with open(input_json_path, "r", encoding="utf-8") as f:
        jobs_list = json.load(f)
        
    session = requests.Session(impersonate="chrome124")
    results = []
    
    print(f"[*] Bắt đầu crawl chi tiết cho {len(jobs_list)} jobs từ TopDev...")
    for job in tqdm(jobs_list, desc="TopDev Details"):
        try:
            detail = parse_topdev_detail(session, job)
            if detail:
                results.append(detail)
        except Exception as e:
            print(f"[!] Lỗi khi cào job {job.get('source_job_id')}: {e}")
        time.sleep(random.uniform(*delay_range))
        
    df_details = pd.DataFrame(results)
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    df_details.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    
    print(f"[+] Hoàn tất! Đã lưu {len(df_details)} job details vào {output_json_path}")
    return df_details

In [ ]:
output_dir = os.path.join("..", "data", "raw")
input_json = os.path.join(output_dir, "topdev_jobs_raw.json")
output_json = os.path.join(output_dir, "topdev_job_details_raw.json")
output_csv = os.path.join(output_dir, "topdev_job_details_raw.csv")

df_topdev_details = crawl_all_topdev_details(input_json, output_json, output_csv)
print("Shape:", df_topdev_details.shape)
df_topdev_details.head(3)